# 4. The taxonomy: affinity propagation, plateau, exemplars

**Affinity propagation (AP)** clusters the consensus $S^\star$ by message passing and returns at once the number of
clusters $C$, the partition, and one **exemplar** per cluster: an actual run of the collection. The number of clusters is
an output (the preference is the median of the entries of $S^\star$, its diagonal first set to its largest value). We use the settings of the original implementation of Frey and
Dueck: damping 0.9, convergence after 100 unchanged iterations, at most 1000 iterations.

**Choosing the taxonomy without labels.** Sweeping $(k_{nn},t)$ over 99 settings gives a resolution map. The **plateau** is
the set of settings where $C$ takes its most frequent value $C^\star$; the taxonomy is the **consensus** of the plateau
partitions, and its **stability** is the mean ARI between each plateau partition and the consensus.

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
RES = ROOT / "results" / "epuck"
load = lambda p: pickle.load(open(p, "rb"))

In [ ]:
import run_sweep as RS
from classification import fusion as F
lab, rk, ctrl = RS.load_labels(); D, _ = RS.load_distances()
idx = RS.subset(1, rk); y = lab[idx]
views = F.scale_views(F.rank_normalise(D, F.ALL_FIELDS, idx))
out = F.run(F.rank_normalise(D, F.ALL_FIELDS, idx), knn=7, t=10)
print("C =", len(np.unique(out["labels"])), " exemplars:", out["exemplars"].tolist(), " converged:", out["converged"])
print("scores against the missions:", RS.score(y, out["labels"]))

## The resolution map of Top-1 and its plateau

In [ ]:
import make_figures as M
G = load(PAPER / "grid.pkl")
fig, ax = plt.subplots(figsize=(5.5, 5))
M.resolution_panel(ax, G[1]["results"], "Top-1"); plt.show()

## Plateau consensus, exemplars and stability

In [ ]:
pc = F.plateau_consensus(G[1]["results"], len(y))
print(f"C* = {pc['C']} on {pc['n_plateau']}/99 settings, stability = {pc['stability']:.3f}")
print("scores of the consensus against the missions:", RS.score(y, pc["labels"]))
order = np.argsort(pc["labels"], kind="stable")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(pc["coassociation"][np.ix_(order, order)], cmap=L.SEQ); axes[0].set_title("co-association on the plateau")
d = load(PAPER / "top1_tracks.pkl")
for k, e in enumerate(pc["exemplars"]):
    ax = fig.add_axes([0.55 + 0.085 * k, 0.3, 0.08, 0.4]); M.draw_tracks(ax, d["tracks"][e]["pos"], L.MCOL[int(y[e])], every=8)
    ax.set_title(f"{k+1}\n{pc['exemplar_frequency'][k]:.0%}", fontsize=8)
axes[1].axis("off"); plt.show()

The percentage under each exemplar is how often that run is AP's exemplar on the plateau.

## All datasets: the taxonomy table

Pooled datasets keep a high **homogeneity** (no cluster mixes missions) while **completeness** decreases: missions are
divided into sub-behaviours, some of which coincide with single controllers. Completeness and ARI compare the partition with the five missions,
so they decrease by construction when sub-behaviours are discovered; on Top-1 (one controller per mission) every score
is expected to reach one.

In [ ]:
T = load(PAPER / "taxonomy.pkl")
for top, t in T.items():
    print(f"Top-{top:<2} C*={t['C']:>2} plateau={t['n_plateau']:>2}/99 stab={t['stability']:.3f} purity={t['purity']:.3f} "
          f"hom={t['homogeneity']:.3f} comp={t['completeness']:.3f} ARI={t['ari']:.3f}")

## Pooled controllers

Mission-by-cluster tables of the consensus on Top-7 and Top-10: block-diagonal, each mission divided into a few clusters.

In [ ]:
M.fig_pooled()
from IPython.display import Image, display
display(Image(filename=str(ROOT / "figures/figA_pooled.png"), width=900))